# Football Player Market Value Interactive Dashboard

**Final Year Dissertation · Big 5 Leagues · 2020–2025**

This notebook is a fully interactive dashboard.

| Section | What you can explore |
|---|---|
| Overview | Market value distributions, top players, league breakdown |
| League Comparison | Filter by season & position; compare leagues side-by-side |
| Position Comparison | Radar charts, age vs value, defensive stats |
| Season Trends | Year-on-year market value & stats changes |
| Player Profile | Photo · career stats · trend chart · Find Similar Player |

## Setup

In [1]:
# Install any missing packages (only runs once)
import subprocess, sys
required = ['plotly', 'ipywidgets']
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  Done.')
print('All packages ready.')

All packages ready.


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings, os
warnings.filterwarnings('ignore')

# Constants
SEASONS = ['2020-2021','2021-2022','2022-2023','2023-2024','2024-2025']
SEASON_LABELS = {'2020-2021':'20/21','2021-2022':'21/22','2022-2023':'22/23',
                 '2023-2024':'23/24','2024-2025':'24/25'}
LEAGUES = ['Premier League','La Liga','Bundesliga','Serie A','Ligue 1']
POSITIONS = ['GK','DF','MF','FW']
LC = {'Premier League':'#7b2d8b','La Liga':'#e63946','Bundesliga':'#e63000',
      'Serie A':'#4361ee','Ligue 1':'#3a86ff'}
PC = {'GK':'#2ecc71','DF':'#3498db','MF':'#9b59b6','FW':'#e74c3c'}

# Plotly template
TEMPLATE = 'plotly_dark'
print('Imports complete.')

Imports complete.


## Data Loading

In [3]:
# Locate app_data.csv relative to this notebook
nb_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
candidates = [
    os.path.join(nb_dir, '..', 'data', 'processed', 'app_data.csv'),
    os.path.join(nb_dir, 'data', 'processed', 'app_data.csv'),
    '../data/processed/app_data.csv',
    'data/processed/app_data.csv',
]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError('Could not find app_data.csv. Run this notebook from the project root or notebooks/ folder.')

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

# Derived columns
df_raw['mv_M'] = df_raw['market_value_end'] / 1e6
df_raw['season_label'] = df_raw['season'].map(SEASON_LABELS)
df_raw['rating_imp'] = df_raw['rating'].copy()
for p in df_raw['pos_enc'].dropna().unique():
    mask = df_raw['pos_enc'] == p
    med  = df_raw.loc[mask, 'rating'].median()
    df_raw.loc[mask & df_raw['rating'].isna(), 'rating_imp'] = med
df_raw['rating_imp'] = df_raw['rating_imp'].fillna(df_raw['rating'].median())

DF = df_raw.copy()  # master reference
print(f'Loaded {len(DF):,} player-season records across {DF["name"].nunique():,} players.')
print(f'Seasons: {sorted(DF["season"].unique())}')
print(f'Image URL coverage: {DF["image_url"].notna().mean():.1%}')

Loaded 13,605 player-season records across 5,595 players.
Seasons: ['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']
Image URL coverage: 92.6%


## Helper Functions

In [4]:
def fmt_eur(v):
    if pd.isna(v): return 'N/A'
    if v >= 1e6:  return f'€{v/1e6:.1f}M'
    if v >= 1e3:  return f'€{v/1e3:.0f}K'
    return f'€{v:.0f}'

def player_card_html(row):
    """Return HTML string for a player info card."""
    img_src = row.get('image_url', '')
    img_html = (f'<img src="{img_src}" width="130" style="border-radius:65px;'
                f'object-fit:cover;border:3px solid #444;"/>'
                if pd.notna(img_src) and isinstance(img_src, str)
                else '<div style="width:130px;height:130px;background:#333;border-radius:65px;'
                     'display:flex;align-items:center;justify-content:center;font-size:50px;">👤</div>')
    pos_full = {'GK':'Goalkeeper','DF':'Defender','MF':'Midfielder','FW':'Forward'}
    def safe(v, fmt='{}'): return fmt.format(v) if pd.notna(v) else 'N/A'
    html = f'''
    <div style="display:flex;gap:24px;padding:20px;background:#1a1c24;
                border-radius:12px;border:1px solid #333;margin-bottom:12px;">
      <div style="flex-shrink:0">{img_html}</div>
      <div style="color:#f0f0f0;">
        <h2 style="margin:0 0 4px 0;color:#fff;">{row['name']}</h2>
        <p style="margin:2px 0;color:#aaa;">{row.get('club','')}&nbsp;&nbsp;·&nbsp;&nbsp;
           {row.get('league','')} &nbsp;&nbsp;·&nbsp;&nbsp; {SEASON_LABELS.get(row.get('season',''),'')}</p>
        <hr style="border-color:#333;margin:8px 0;"/>
        <table style="border-collapse:collapse;color:#ddd;font-size:14px;">
          <tr><td style="padding:2px 16px 2px 0;"> Position</td>
              <td><b>{pos_full.get(row.get('pos_label',''),'')}</b></td></tr>
          <tr><td> Age</td><td><b>{safe(row.get('Age'), '{:.0f}')}</b></td></tr>
          <tr><td> Nationality</td><td><b>{row.get('country_of_birth','N/A')}</b></td></tr>
          <tr><td> Height</td><td><b>{safe(row.get('height_in_cm'), '{:.0f} cm')}</b></td></tr>
          <tr><td> Foot</td><td><b>{str(row.get('foot','')).capitalize() if pd.notna(row.get('foot')) else 'N/A'}</b></td></tr>
          <tr><td> Team Pos</td><td><b>{safe(row.get('team_finishing_pos'), '{:.0f}')}</b></td></tr>
        </table>
      </div>
      <div style="margin-left:auto;text-align:right;color:#f0f0f0;">
        <div style="font-size:13px;color:#aaa;">Market Value</div>
        <div style="font-size:28px;font-weight:bold;color:#e15759;">{fmt_eur(row.get('market_value_end'))}</div>
        <hr style="border-color:#333;margin:8px 0;"/>
        <table style="text-align:right;color:#ddd;font-size:14px;margin-left:auto;">
          <tr><td style="padding:2px 0 2px 16px;">Goals</td><td style="padding-left:12px;"><b>{safe(row.get('goals'),'{:.0f}')}</b></td></tr>
          <tr><td>Assists</td><td><b>{safe(row.get('assists'),'{:.0f}')}</b></td></tr>
          <tr><td>xG / 90</td><td><b>{safe(row.get('xG90'),'{:.2f}')}</b></td></tr>
          <tr><td>xA / 90</td><td><b>{safe(row.get('xA90'),'{:.2f}')}</b></td></tr>
          <tr><td>Rating</td><td><b>{safe(row.get('rating_imp'),'{:.2f}')}</b></td></tr>
          <tr><td>Appearances</td><td><b>{safe(row.get('appearances'),'{:.0f}')}</b></td></tr>
        </table>
      </div>
    </div>'''
    return html

SIM_FEATS = ['Age','Gls_90','Ast_90','G+A_90','xG90','xA90',
             'xGChain90','xGBuildup90','rating_imp','minutes','team_finishing_pos']

def find_similar(df_pool, player_name, season, n=5, same_pos=True):
    """Cosine similarity on normalised feature vector."""
    feats = [f for f in SIM_FEATS if f in df_pool.columns]
    pool  = df_pool.copy().reset_index(drop=True)
    pool[feats] = pool[feats].fillna(pool[feats].mean())
    target_rows = pool[(pool['name'] == player_name) & (pool['season'] == season)]
    if target_rows.empty: return pd.DataFrame()
    p_idx = target_rows.index[0]
    if same_pos:
        pos = pool.loc[p_idx, 'pos_label']
        pool = pool[pool['pos_label'] == pos].reset_index(drop=True)
        t_rows = pool[(pool['name'] == player_name) & (pool['season'] == season)]
        if t_rows.empty: return pd.DataFrame()
        p_idx = t_rows.index[0]
    X = pool[feats].values.astype(float)
    mu = X.mean(0); sig = X.std(0) + 1e-9
    Xs = (X - mu) / sig
    tv = Xs[p_idx]
    norms = np.linalg.norm(Xs, axis=1) + 1e-9
    sims  = Xs @ tv / (norms * (np.linalg.norm(tv) + 1e-9))
    sims[p_idx] = -1
    top = np.argsort(sims)[::-1][:n]
    out = pool.iloc[top][['name','club','league','season','pos_label',
                           'Age','goals','assists','rating_imp',
                           'market_value_end','image_url']].copy()
    out['similarity'] = sims[top]
    return out

def similar_cards_html(sim_df):
    if sim_df.empty: return '<p style="color:#aaa">No results.</p>'
    cards = ''
    for _, r in sim_df.iterrows():
        img = (f'<img src="{r["image_url"]}" width="70" style="border-radius:35px;'
               f'object-fit:cover;"/>'
               if pd.notna(r.get('image_url')) else '👤')
        pct = int(r['similarity'] * 100)
        bar = f'<div style="background:#333;border-radius:4px;height:6px;margin-top:4px;">'\
              f'<div style="background:#e15759;width:{pct}%;height:100%;border-radius:4px;"></div></div>'
        cards += f'''
        <div style="display:inline-flex;flex-direction:column;align-items:center;
                    width:140px;padding:12px;margin:6px;background:#1e2030;
                    border-radius:10px;border:1px solid #333;vertical-align:top;">
          {img}
          <div style="font-size:13px;font-weight:bold;color:#fff;text-align:center;
                      margin-top:6px;word-break:break-word;">{r['name']}</div>
          <div style="font-size:11px;color:#aaa;text-align:center;">
            {r['pos_label']} · {r.get('league','')}</div>
          <div style="font-size:11px;color:#aaa;">{SEASON_LABELS.get(r['season'],r['season'])}</div>
          <div style="font-size:12px;color:#e15759;font-weight:bold;">{fmt_eur(r['market_value_end'])}</div>
          <div style="font-size:11px;color:#aaa;"> {int(r['goals']) if pd.notna(r['goals']) else '?'} · '
                {int(r['assists']) if pd.notna(r['assists']) else '?'}</div>
          {bar}
          <div style="font-size:10px;color:#888;">{pct}% similar</div>
        </div>'''
    return f'<div style="display:flex;flex-wrap:wrap;gap:4px;">{cards}</div>'

print('Helper functions ready.')

Helper functions ready.


## Section 1: Overview

In [5]:
# KPI summary
kpis = [
    ('Players', f"{DF['name'].nunique():,}"),
    ('Season Records', f"{len(DF):,}"),
    ('Leagues', '5'),
    ('Seasons', '5'),
    ('Avg Market Value', fmt_eur(DF['market_value_end'].mean())),
    ('Median Market Value', fmt_eur(DF['market_value_end'].median())),
]
kpi_html = '<div style="display:flex;gap:16px;flex-wrap:wrap;margin-bottom:20px;">'
for label, val in kpis:
    kpi_html += f'''
    <div style="background:#1a1c24;border:1px solid #333;border-radius:10px;
                padding:16px 24px;min-width:140px;text-align:center;">
      <div style="font-size:11px;color:#aaa;text-transform:uppercase;">{label}</div>
      <div style="font-size:24px;font-weight:bold;color:#e15759;">{val}</div>
    </div>'''
kpi_html += '</div>'
display(HTML(kpi_html))

In [6]:
# Market value by league (box plot)
fig = px.box(DF.dropna(subset=['mv_M']), x='league', y='mv_M',
             color='league', color_discrete_map=LC,
             template=TEMPLATE,
             labels={'mv_M':'Market Value (€M)','league':''},
             title='Market Value Distribution by League',
             category_orders={'league': LEAGUES})
fig.update_traces(showlegend=False)
fig.update_layout(height=380)
fig.show()

In [7]:
# Avg market value trend
mv_t = DF.groupby(['season_label','league'])['mv_M'].mean().reset_index()
fig2 = px.line(mv_t, x='season_label', y='mv_M', color='league',
               color_discrete_map=LC, markers=True, template=TEMPLATE,
               labels={'mv_M':'Avg Market Value (€M)','season_label':'Season','league':'League'},
               title='Average Market Value by Season',
               category_orders={'season_label':list(SEASON_LABELS.values()),'league':LEAGUES})
fig2.update_layout(height=380, legend=dict(orientation='h', y=-0.2))
fig2.show()

In [8]:
# Top 15 players by peak market value
top15 = (DF.groupby('name')['market_value_end'].max()
           .sort_values(ascending=False).head(15).reset_index())
top15['mv_M'] = top15['market_value_end'] / 1e6
top15 = top15.merge(DF[['name','pos_label']].drop_duplicates('name'), on='name', how='left')
fig3 = px.bar(top15, x='mv_M', y='name', orientation='h',
              color='pos_label', color_discrete_map=PC, template=TEMPLATE,
              labels={'mv_M':'Peak Market Value (€M)','name':''},
              title='Top 15 Players by Peak Market Value')
fig3.update_layout(height=450, yaxis=dict(autorange='reversed'),
                   legend=dict(orientation='h', y=-0.15))
fig3.show()

## Section 2: League Comparison

Use the filters below to compare leagues.

In [9]:
# Widgets
w_lg_seasons = widgets.SelectMultiple(
    options=SEASONS, value=SEASONS,
    description='Seasons:', rows=5,
    style={'description_width': '70px'},
    layout=widgets.Layout(width='220px'),
)
w_lg_seasons.format_value = lambda v: SEASON_LABELS[v]
w_lg_pos = widgets.SelectMultiple(
    options=POSITIONS, value=POSITIONS,
    description='Positions:', rows=4,
    style={'description_width': '70px'},
    layout=widgets.Layout(width='180px'),
)
w_lg_stat = widgets.Dropdown(
    options=[('Goals','goals'),('Assists','assists'),('Goals + Assists','G+A'),
             ('Goals / 90','Gls_90'),('Assists / 90','Ast_90'),
             ('xG / 90','xG90'),('xA / 90','xA90'),
             ('xG Chain / 90','xGChain90'),('Player Rating','rating_imp'),
             ('Appearances','appearances'),('Minutes Played','minutes')],
    value='xG90', description='Stat:', style={'description_width':'50px'},
    layout=widgets.Layout(width='260px')
)
lg_out = widgets.Output()

def update_league(*_):
    sel_s = list(w_lg_seasons.value)
    sel_p = list(w_lg_pos.value)
    stat  = w_lg_stat.value
    if not sel_s or not sel_p:
        return
    sub = DF[DF['season'].isin(sel_s) & DF['pos_label'].isin(sel_p)]
    with lg_out:
        clear_output(wait=True)
        display(HTML(f'<p style="color:#aaa">📋 {len(sub):,} records in selection</p>'))

        # Violin: market value
        fig1 = px.violin(sub.dropna(subset=['mv_M']), x='league', y='mv_M',
                         color='league', color_discrete_map=LC, box=True, points=False,
                         template=TEMPLATE,
                         labels={'mv_M':'Market Value (€M)','league':''},
                         title='Market Value Distribution by League',
                         category_orders={'league':LEAGUES})
        fig1.update_traces(showlegend=False)
        fig1.update_layout(height=370)
        fig1.show()

        # Bar: selected stat
        stat_agg = sub.groupby('league')[stat].mean().reset_index().sort_values(stat, ascending=False)
        stat_label = dict(w_lg_stat.options)[stat] if stat in dict(w_lg_stat.options).values() else stat
        stat_label = next((k for k,v in w_lg_stat.options if v==stat), stat)
        fig2 = px.bar(stat_agg, x='league', y=stat, color='league',
                      color_discrete_map=LC, template=TEMPLATE,
                      labels={stat: f'Avg {stat_label}', 'league':''},
                      title=f'Average {stat_label} by League',
                      category_orders={'league':LEAGUES})
        fig2.update_traces(showlegend=False)
        fig2.update_layout(height=340)
        fig2.show()

        # Scatter: xG vs Market Value
        samp = sub.dropna(subset=['xG90','mv_M']).sample(min(2000, len(sub)), random_state=42)
        fig3 = px.scatter(samp, x='xG90', y='mv_M', color='league',
                          color_discrete_map=LC, opacity=0.5, template=TEMPLATE,
                          hover_data=['name','season_label'],
                          trendline='ols',
                          labels={'xG90':'xG per 90','mv_M':'Market Value (€M)','league':'League'},
                          title='xG/90 vs Market Value by League',
                          category_orders={'league':LEAGUES})
        fig3.update_layout(height=370, legend=dict(orientation='h', y=-0.2))
        fig3.show()

        # Top 10 table per league
        display(HTML('<h4 style="color:#ddd">Top 5 Players by Market Value per League</h4>'))
        rows_html = '<div style="display:flex;gap:12px;flex-wrap:wrap;">'
        for lg in LEAGUES:
            top = (sub[sub['league']==lg]
                   .sort_values('market_value_end', ascending=False)
                   .drop_duplicates('name').head(5)
                   [['name','pos_label','club','season_label','mv_M']])
            rows = ''.join(f'<tr><td style="padding:2px 8px;">{r["name"]}</td>'
                           f'<td style="padding:2px 8px;color:#aaa">{r["pos_label"]}</td>'
                           f'<td style="padding:2px 8px;color:#e15759">€{r["mv_M"]:.0f}M</td></tr>'
                           for _, r in top.iterrows())
            rows_html += (f'<div style="background:#1a1c24;border:1px solid #333;'
                          f'border-radius:8px;padding:12px;min-width:200px;">'
                          f'<div style="font-weight:bold;color:#ddd;margin-bottom:6px;">{lg}</div>'
                          f'<table style="color:#ccc;font-size:13px;">{rows}</table></div>')
        rows_html += '</div>'
        display(HTML(rows_html))

for w in [w_lg_seasons, w_lg_pos, w_lg_stat]:
    w.observe(update_league, names='value')

controls = widgets.HBox([w_lg_seasons, w_lg_pos, w_lg_stat],
                         layout=widgets.Layout(gap='20px', align_items='flex-start'))
display(controls, lg_out)
update_league()

Output()

## Section 3: Position Comparison

In [10]:
w_pos_leagues = widgets.SelectMultiple(
    options=LEAGUES, value=LEAGUES, description='Leagues:', rows=5,
    style={'description_width':'70px'}, layout=widgets.Layout(width='220px')
)
w_pos_seasons = widgets.SelectMultiple(
    options=SEASONS, value=SEASONS, description='Seasons:', rows=5,
    style={'description_width':'70px'}, layout=widgets.Layout(width='220px')
)
pos_out = widgets.Output()

def update_position(*_):
    sel_l = list(w_pos_leagues.value)
    sel_s = list(w_pos_seasons.value)
    if not sel_l or not sel_s: return
    sub = DF[DF['league'].isin(sel_l) & DF['season'].isin(sel_s)]
    with pos_out:
        clear_output(wait=True)
        display(HTML(f'<p style="color:#aaa">📋 {len(sub):,} records</p>'))

        # Box: market value by position
        fig1 = px.box(sub.dropna(subset=['mv_M']), x='pos_label', y='mv_M',
                      color='pos_label', color_discrete_map=PC, template=TEMPLATE,
                      labels={'mv_M':'Market Value (€M)','pos_label':'Position'},
                      title='Market Value by Position',
                      category_orders={'pos_label':POSITIONS})
        fig1.update_traces(showlegend=False)
        fig1.update_layout(height=360)
        fig1.show()

        # Radar chart: avg stats by position
        stat_cols = ['goals','assists','xG90','xA90','xGBuildup90','Gls_90','Ast_90','rating_imp']
        stat_lbls = ['Goals','Assists','xG/90','xA/90','xG Buildup/90','Gls/90','Ast/90','Rating']
        pos_stats = sub.groupby('pos_label')[stat_cols].mean()
        pos_n = pos_stats.copy()
        for c in stat_cols:
            mn, mx = pos_stats[c].min(), pos_stats[c].max()
            pos_n[c] = (pos_stats[c] - mn) / max(mx - mn, 1e-9)
        fig2 = go.Figure()
        for pos in POSITIONS:
            if pos not in pos_n.index: continue
            vals = [pos_n.loc[pos, c] for c in stat_cols]
            fig2.add_trace(go.Scatterpolar(
                r=vals + [vals[0]], theta=stat_lbls + [stat_lbls[0]],
                fill='toself', name=pos,
                line=dict(color=PC.get(pos, 'grey'))))
        fig2.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
            template=TEMPLATE, height=420, title='Normalised Average Stats by Position',
            legend=dict(orientation='h', y=-0.1)
        )
        fig2.show()

        # Age vs MV scatter
        samp = sub.dropna(subset=['Age','mv_M']).sample(min(2000,len(sub)), random_state=7)
        fig3 = px.scatter(samp, x='Age', y='mv_M', color='pos_label',
                          color_discrete_map=PC, opacity=0.45, template=TEMPLATE,
                          hover_data=['name','league','season_label'], trendline='lowess',
                          labels={'Age':'Age','mv_M':'Market Value (€M)','pos_label':'Position'},
                          title='Age vs Market Value by Position',
                          category_orders={'pos_label':POSITIONS})
        fig3.update_layout(height=370, legend=dict(orientation='h', y=-0.2))
        fig3.show()

        # GK defensive stats
        gk = sub[sub['pos_label']=='GK'].dropna(subset=['gk_clean_sheet_pct','mv_M'])
        if len(gk) > 10:
            fig4 = px.scatter(gk, x='gk_clean_sheet_pct', y='mv_M', template=TEMPLATE,
                              color_discrete_sequence=[PC['GK']], opacity=0.7,
                              hover_data=['name','season_label','club'], trendline='ols',
                              labels={'gk_clean_sheet_pct':'Clean Sheet %','mv_M':'Market Value (€M)'},
                              title='GK: Clean Sheet % vs Market Value')
            fig4.update_layout(height=340)
            fig4.show()

for w in [w_pos_leagues, w_pos_seasons]:
    w.observe(update_position, names='value')

display(widgets.HBox([w_pos_leagues, w_pos_seasons],
                      layout=widgets.Layout(gap='20px')), pos_out)
update_position()

Output()

## Section 4: Season Trends (Year-on-Year)

In [11]:
w_yoy_leagues = widgets.SelectMultiple(
    options=LEAGUES, value=LEAGUES, description='Leagues:', rows=5,
    style={'description_width':'70px'}, layout=widgets.Layout(width='220px')
)
w_yoy_pos = widgets.SelectMultiple(
    options=POSITIONS, value=POSITIONS, description='Positions:', rows=4,
    style={'description_width':'70px'}, layout=widgets.Layout(width='180px')
)
w_yoy_stat = widgets.Dropdown(
    options=[('Goals','goals'),('Assists','assists'),('xG / 90','xG90'),
             ('xA / 90','xA90'),('Player Rating','rating_imp'),
             ('Appearances','appearances'),('Minutes','minutes')],
    value='xG90', description='Trend stat:',
    style={'description_width':'80px'}, layout=widgets.Layout(width='260px')
)
yoy_out = widgets.Output()
season_order = list(SEASON_LABELS.values())

def update_yoy(*_):
    sel_l = list(w_yoy_leagues.value)
    sel_p = list(w_yoy_pos.value)
    stat  = w_yoy_stat.value
    if not sel_l or not sel_p: return
    sub = DF[DF['league'].isin(sel_l) & DF['pos_label'].isin(sel_p)]
    with yoy_out:
        clear_output(wait=True)

        # Median MV trend
        mv_t = sub.groupby(['season_label','league'])['mv_M'].median().reset_index()
        fig1 = px.line(mv_t, x='season_label', y='mv_M', color='league',
                       color_discrete_map=LC, markers=True, template=TEMPLATE,
                       labels={'mv_M':'Median Market Value (€M)','season_label':'Season','league':'League'},
                       title='Median Market Value — Year on Year',
                       category_orders={'season_label':season_order,'league':LEAGUES})
        fig1.update_layout(height=370, legend=dict(orientation='h', y=-0.2))
        fig1.show()

        # YoY % change
        yoy_list = []
        for lg in sel_l:
            vals = (sub[sub['league']==lg]
                    .groupby('season')['market_value_end'].median()
                    .reindex(SEASONS))
            for i in range(1, len(SEASONS)):
                prev, curr = vals.iloc[i-1], vals.iloc[i]
                if pd.notna(prev) and pd.notna(curr) and prev > 0:
                    trans = f"{SEASON_LABELS[SEASONS[i-1]]}→{SEASON_LABELS[SEASONS[i]]}"
                    yoy_list.append({'League':lg,'Transition':trans,
                                     'Change (%)': (curr-prev)/prev*100})
        if yoy_list:
            yoy_df = pd.DataFrame(yoy_list)
            fig2 = px.bar(yoy_df, x='Transition', y='Change (%)', color='League',
                          color_discrete_map=LC, barmode='group', template=TEMPLATE,
                          title='Year-on-Year % Change in Median Market Value',
                          category_orders={'League':LEAGUES})
            fig2.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.3)
            fig2.update_layout(height=370, legend=dict(orientation='h', y=-0.2))
            fig2.show()

        # Stat trend
        stat_t = sub.groupby(['season_label','league'])[stat].mean().reset_index()
        stat_lbl = next((k for k,v in w_yoy_stat.options if v==stat), stat)
        fig3 = px.line(stat_t, x='season_label', y=stat, color='league',
                       color_discrete_map=LC, markers=True, template=TEMPLATE,
                       labels={stat:f'Avg {stat_lbl}','season_label':'Season','league':'League'},
                       title=f'{stat_lbl} — Year on Year',
                       category_orders={'season_label':season_order,'league':LEAGUES})
        fig3.update_layout(height=370, legend=dict(orientation='h', y=-0.2))
        fig3.show()

        # Position value trends
        pos_t = sub.groupby(['season_label','pos_label'])['mv_M'].median().reset_index()
        fig4 = px.line(pos_t, x='season_label', y='mv_M', color='pos_label',
                       color_discrete_map=PC, markers=True, template=TEMPLATE,
                       labels={'mv_M':'Median MV (€M)','season_label':'Season','pos_label':'Position'},
                       title='Median Market Value by Position — Year on Year',
                       category_orders={'season_label':season_order,'pos_label':POSITIONS})
        fig4.update_layout(height=370, legend=dict(orientation='h', y=-0.2))
        fig4.show()

for w in [w_yoy_leagues, w_yoy_pos, w_yoy_stat]:
    w.observe(update_yoy, names='value')

display(widgets.HBox([w_yoy_leagues, w_yoy_pos, w_yoy_stat],
                      layout=widgets.Layout(gap='20px', align_items='flex-start')), yoy_out)
update_yoy()

Output()

## Section 5: Player Profile

Search for any player. Use the Season slider to view a specific year. The Find Similar Players section uses cosine similarity on normalised performance stats.

In [12]:
all_names = sorted(DF['name'].dropna().unique())

w_player = widgets.Combobox(
    placeholder='Type a player name (e.g. Mohamed Salah)',
    options=all_names,
    ensure_option=True,
    description='Player:',
    style={'description_width':'60px'},
    layout=widgets.Layout(width='380px')
)
w_season = widgets.Dropdown(
    options=SEASONS, value=SEASONS[-1],
    description='Season:',
    style={'description_width':'60px'},
    layout=widgets.Layout(width='200px'),
)
w_season.observe(lambda c: None, names='value')  # placeholder

# Trend stat selector
w_trend_stats = widgets.SelectMultiple(
    options=[('Goals','goals'),('Assists','assists'),('xG/90','xG90'),
             ('xA/90','xA90'),('Rating','rating_imp'),
             ('Market Value (€M)','mv_M'),('Appearances','appearances')],
    value=['goals','assists','mv_M'],
    description='Trend:', rows=4,
    style={'description_width':'60px'},
    layout=widgets.Layout(width='260px')
)

# Similar players options
w_same_pos  = widgets.Checkbox(value=True,  description='Same position only')
w_same_seas = widgets.Checkbox(value=True,  description='Same season only')
w_n_sim     = widgets.IntSlider(value=6, min=3, max=12, step=1,
                                description='Show:', style={'description_width':'50px'},
                                layout=widgets.Layout(width='300px'))

profile_out = widgets.Output()
trend_out   = widgets.Output()
similar_out = widgets.Output()

def update_season_options(player_name):
    """Update season dropdown to only show seasons the player appears in."""
    rows = DF[DF['name'] == player_name]
    avail = sorted(rows['season'].unique())
    w_season.options = avail if avail else SEASONS
    if w_season.value not in w_season.options:
        w_season.value = avail[-1] if avail else SEASONS[-1]

def render_profile(*_):
    pname  = w_player.value
    season = w_season.value
    if not pname or pname not in all_names:
        with profile_out:
            clear_output(wait=True)
            display(HTML('<p style="color:#aaa">Start typing a player name above.</p>'))
        return

    update_season_options(pname)
    season = w_season.value  # may have changed

    # Profile card
    rows = DF[(DF['name'] == pname) & (DF['season'] == season)]
    if rows.empty:
        with profile_out:
            clear_output(wait=True)
            display(HTML(f'<p style="color:#aaa">No data for {pname} in {season}.</p>'))
        return
    p = rows.iloc[0]

    with profile_out:
        clear_output(wait=True)
        display(HTML(player_card_html(p)))
        # GK / DF position-specific stats
        if p.get('pos_label') == 'GK':
            gk_html = ('<div style="display:flex;gap:16px;margin-top:8px;">'
                       + ''.join(f'<div style="background:#1a1c24;border:1px solid #333;'
                                 f'border-radius:8px;padding:12px 20px;text-align:center;">'
                                 f'<div style="color:#aaa;font-size:11px;">{lbl}</div>'
                                 f'<div style="color:#e15759;font-size:20px;font-weight:bold;">{val}</div></div>'
                                 for lbl, val in [
                                     ('Clean Sheet %', f"{p['gk_clean_sheet_pct']:.1f}%" if pd.notna(p.get('gk_clean_sheet_pct')) else 'N/A'),
                                     ('Goals Conceded/90', f"{p['gk_goals_conceded_per90']:.2f}" if pd.notna(p.get('gk_goals_conceded_per90')) else 'N/A'),
                                     ('Win Rate', f"{p['gk_win_rate']:.1f}%" if pd.notna(p.get('gk_win_rate')) else 'N/A'),
                                 ]) + '</div>')
            display(HTML(gk_html))
        elif p.get('pos_label') == 'DF':
            df_html = ('<div style="display:flex;gap:16px;margin-top:8px;">'
                       + ''.join(f'<div style="background:#1a1c24;border:1px solid #333;'
                                 f'border-radius:8px;padding:12px 20px;text-align:center;">'
                                 f'<div style="color:#aaa;font-size:11px;">{lbl}</div>'
                                 f'<div style="color:#3498db;font-size:20px;font-weight:bold;">{val}</div></div>'
                                 for lbl, val in [
                                     ('Clean Sheet %', f"{p['def_clean_sheet_pct']:.1f}%" if pd.notna(p.get('def_clean_sheet_pct')) else 'N/A'),
                                     ('Goals Conceded/90', f"{p['def_goals_conceded_per90']:.2f}" if pd.notna(p.get('def_goals_conceded_per90')) else 'N/A'),
                                     ('xG Buildup/90', f"{p['xGBuildup90']:.2f}" if pd.notna(p.get('xGBuildup90')) else 'N/A'),
                                 ]) + '</div>')
            display(HTML(df_html))

    # Career trend chart
    with trend_out:
        clear_output(wait=True)
        player_rows = DF[DF['name'] == pname].sort_values('season')
        if len(player_rows) > 1:
            sel_stats = list(w_trend_stats.value)
            stat_lbls_map = dict(w_trend_stats.options)
            fig = go.Figure()
            palette = px.colors.qualitative.Set2
            for i, stat in enumerate(sel_stats):
                if stat in player_rows.columns:
                    fig.add_trace(go.Scatter(
                        x=player_rows['season_label'], y=player_rows[stat],
                        mode='lines+markers', name=stat_lbls_map.get(stat, stat),
                        line=dict(color=palette[i % len(palette)], width=3),
                        marker=dict(size=9)
                    ))
            # Highlight selected season
            sel_lbl = SEASON_LABELS.get(season, season)
            fig.add_vline(x=sel_lbl, line_dash='dot', line_color='#e15759',
                          opacity=0.7, annotation_text='Selected season',
                          annotation_font_color='#e15759')
            fig.update_layout(
                title=f'{pname} — Career Trend',
                template=TEMPLATE, height=360,
                legend=dict(orientation='h', y=-0.25),
                xaxis_title='Season'
            )
            fig.show()
        else:
            display(HTML('<p style="color:#aaa">Only one season of data available — no trend to show.</p>'))

    # Similar players
    with similar_out:
        clear_output(wait=True)
        pool = DF[DF['season'] == season].copy() if w_same_seas.value else DF.copy()
        sim = find_similar(pool, pname, season, n=w_n_sim.value, same_pos=w_same_pos.value)
        pos_note = 'same position' if w_same_pos.value else 'all positions'
        seas_note = 'same season' if w_same_seas.value else 'all seasons'
        display(HTML(f'<h4 style="color:#ddd">Most similar to {pname} ({pos_note}, {seas_note})</h4>'))
        display(HTML(similar_cards_html(sim)))

# Wire up all widgets
for w in [w_player, w_season, w_same_pos, w_same_seas, w_n_sim]:
    w.observe(render_profile, names='value')
w_trend_stats.observe(lambda _: render_profile(), names='value')

# Layout
search_row = widgets.HBox([w_player, w_season],
                           layout=widgets.Layout(gap='16px', align_items='center'))
trend_controls = widgets.HBox(
    [widgets.Label('Career trend stats:'), w_trend_stats],
    layout=widgets.Layout(align_items='center', gap='12px')
)
sim_controls = widgets.HBox(
    [w_same_pos, w_same_seas, w_n_sim],
    layout=widgets.Layout(gap='20px', align_items='center')
)

display(
    widgets.HTML('<h4 style="color:#ddd">Search</h4>'),
    search_row,
    widgets.HTML('<hr style="border-color:#333"/>'),
    profile_out,
    widgets.HTML('<h4 style="color:#ddd">Career Trend</h4>'),
    trend_controls,
    trend_out,
    widgets.HTML('<hr style="border-color:#333"/>'),
    widgets.HTML('<h4 style="color:#ddd">Find Similar Players</h4>'),
    sim_controls,
    similar_out,
)

# Default: Mohamed Salah
w_player.value = 'Mohamed Salah'
render_profile()

HTML(value='<h4 style="color:#ddd">Search</h4>')

HTML(value='<hr style="border-color:#333"/>')

Output()

HTML(value='<h4 style="color:#ddd">Career Trend</h4>')

Output()

HTML(value='<hr style="border-color:#333"/>')

HTML(value='<h4 style="color:#ddd">Find Similar Players</h4>')

Output()